# TOTVS Conversational Intelligence — Análise NLP

**Challenge FIAP x TOTVS 2026**  
Notebook de análise textual das transcrições de reuniões comerciais usando TF-IDF, limpeza de texto, análise exploratória, embeddings e geração de insights.

GRUPO:

Bruno Gama Masseiras              | RM562526

Kauai Rosa de Assis Rocha         | RM563256

Lucas Gilmar Pedro da Silva       | RM566231

Leandro Afonso Silva Santos Junior | RM561344

Murilo da Silva Basto             | RM565445

---

## Sumário
1. [Explicação do Projeto](#01)
2. [Carregamento e Limpeza dos Dados](#02)
3. [Aplicação de TF-IDF](#03)
4. [Análise Exploratória — Gráficos e Padrões](#04)
5. [Embeddings vs TF-IDF](#05)
6. [Insights — Oportunidades, Riscos e Sugestões](#06)

---
## 01 — Explicação do Projeto <a id='01'></a>

### O que é o TOTVS Conversational Intelligence?

A **TOTVS** é a maior empresa de software de gestão (ERP) do Brasil, atendendo mais de 50.000 clientes. Sua força de vendas e Customer Success realiza **milhares de reuniões por dia** com clientes — e cada transcrição contém informações valiosas que costumam se perder ou demorar dias para chegar à liderança comercial.

### Problema
- Transcrições brutas de reuniões não são analisadas sistematicamente
- Riscos de churn (cancelamento) são identificados tarde demais
- Oportunidades de upsell e cross-sell ficam enterradas em texto
- O Diretor Comercial não tem visibilidade em tempo real

### Solução
Plataforma de IA que processa transcrições automaticamente e extrai:

| Output | Descrição |
|---|---|
| **Sentimento** | POSITIVO / NEGATIVO / NEUTRO / MISTO via BERT multilingual |
| **Oportunidades** | Upsell, cross-sell e novos produtos com score 0–1 |
| **Riscos de churn** | Sinais de insatisfação, concorrência e cancelamento |
| **Produtos mapeados** | 15+ produtos TOTVS em uso ou de interesse |
| **Alertas** | Notificações prioritárias com severidade ALTO/MÉDIO/BAIXO |

### Arquitetura Técnica
```
Transcrição (texto/áudio/JSON)
         |
    [LeanIA — Python / FastAPI]
    ├── BERT multilingual (sentimento)
    └── Regras TF-IDF-like (entidades)
         |
    [Backend — Java Spring Boot]
         |
    [MongoDB Atlas]
         |
    [Dashboard — Next.js / Vercel]
```

### Impacto esperado no negócio
- Redução do tempo de identificação de risco de churn: **de dias para segundos**
- Aumento da taxa de conversão de oportunidades por priorização automática
- Escalabilidade: o mesmo sistema analisa 1 ou 10.000 reuniões por dia

In [ ]:
# Instalação das dependências
!pip install scikit-learn pandas matplotlib seaborn wordcloud nltk --quiet

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Estilo dos gráficos
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
CORES = ['#534AB7', '#1D9E75', '#D85A30', '#BA7517', '#185FA5', '#888780']

print('✅ Dependências carregadas com sucesso')

---
## 02 — Carregamento e Limpeza dos Dados <a id='02'></a>

### Dataset
Usamos transcrições sintéticas que representam fielmente o vocabulário real de reuniões de vendas TOTVS — incluindo produtos, módulos, concorrentes e padrões de linguagem identificados no `analyzer.py`.

### Etapas de limpeza (espelhando o `analyzer.py` do projeto)
1. Remoção de caracteres nulos (`\x00`)
2. Colapso de espaços múltiplos via regex
3. Conversão para minúsculo
4. Remoção de pontuação (para TF-IDF; mantida para BERT)
5. Remoção de stopwords em português

In [ ]:
# Dataset de transcrições sintéticas baseado no domínio TOTVS
transcricoes_raw = [
    {
        "id": 1,
        "cliente": "Metalúrgica Horizonte Ltda",
        "vendedor": "Ana Souza",
        "texto": """Bom dia Ana. O nosso Protheus está atendendo bem o backoffice,
        mas o time de RH está sofrendo com a folha manual. O pessoal viu uma demo da Senior
        e gostou bastante. Precisamos resolver isso até o próximo trimestre.
        O nosso budget é de aproximadamente R$ 50 mil para implantação.
        A diretoria aprovou a expansão, mas queremos ver uma proposta concreta."""
    },
    {
        "id": 2,
        "cliente": "Distribuidora Central SP",
        "vendedor": "Carlos Melo",
        "texto": """Carlos, estamos muito frustrados com o suporte. Já abrimos chamado
        há 3 semanas e ninguém resolve. O sistema trava todo dia no faturamento.
        Estamos avaliando outras opções, inclusive o SAP Business One.
        Se isso não for resolvido até o final do mês, vamos cancelar o contrato.
        Nossa operação está parada e estamos perdendo clientes."""
    },
    {
        "id": 3,
        "cliente": "Grupo Educacional Volare",
        "vendedor": "Beatriz Lima",
        "texto": """Boa tarde Beatriz. Usamos o TOTVS Educacional há 5 anos e estamos
        muito satisfeitos com o sistema. A integração com o módulo financeiro
        foi excelente. Gostaríamos de expandir para o módulo de BI e analytics
        para o próximo semestre. Temos verba aprovada de R$ 80 mil.
        Seria possível agendar uma demonstração do TOTVS Carol?"""
    },
    {
        "id": 4,
        "cliente": "Construtora Nova Era",
        "vendedor": "Roberto Dias",
        "texto": """Roberto, o TOTVS Construção está funcionando bem para gestão de obras.
        Porém nosso time de compras está reclamando muito da lentidão no módulo de estoque.
        Temos interesse em implantar o WMS para o próximo ano.
        Mas antes precisamos resolver o problema do e-Social que está dando erro
        e a nota fiscal eletrônica não está emitindo corretamente."""
    },
    {
        "id": 5,
        "cliente": "Farmacêutica BioVida",
        "vendedor": "Juliana Costa",
        "texto": """Juliana, migramos do SAP há 2 anos para o TOTVS Protheus e foi uma
        decisão excelente. A equipe adorou a interface e a produtividade melhorou muito.
        Agora queremos implantar o módulo de qualidade e PCP para a fábrica.
        O CEO aprovou o investimento. Precisamos de uma cotação urgente.
        Temos concorrentes do Sankhya nos procurando mas preferimos o ecossistema TOTVS."""
    },
    {
        "id": 6,
        "cliente": "Logística Expresso Sul",
        "vendedor": "Marcos Ferreira",
        "texto": """Marcos, o contrato vence em 3 meses e ainda não decidimos renovar.
        O TOTVS Winthor não está atendendo nossas necessidades de TMS.
        Recebemos uma proposta da Linx que está bem atraente.
        Nossa operação logística cresceu 40% e o sistema não acompanhou.
        Não renovar é uma opção real se não houver melhorias até lá."""
    },
    {
        "id": 7,
        "cliente": "Saúde Total Hospitais",
        "vendedor": "Fernanda Alves",
        "texto": """Fernanda, o TOTVS Saúde está operacional em todos os 3 hospitais.
        Estamos muito contentes com o resultado. Superou nossas expectativas.
        Gostaríamos de discutir a expansão para o módulo de benefícios e
        recrutamento de pessoal. Temos budget de R$ 120 mil disponível.
        A diretoria aprovou e gostaríamos de iniciar no próximo trimestre."""
    },
    {
        "id": 8,
        "cliente": "Indústria Têxtil Araçá",
        "vendedor": "Paulo Henrique",
        "texto": """Paulo, estamos decepcionados com os erros no módulo de compras.
        O sistema pior do que tínhamos antes. O suporte é ruim, sem retorno.
        Já estamos conversando com a Mega Sistemas como alternativa.
        O nosso contrato é anual e se nada mudar, não renovaremos.
        A situação é crítica e o nosso diretor está insatisfeito."""
    }
]

df = pd.DataFrame(transcricoes_raw)
print(f'Dataset carregado: {len(df)} transcrições')
print(f'Colunas: {list(df.columns)}')
df[['id', 'cliente', 'vendedor']].head()

In [ ]:
# Stopwords em português (expandidas para o domínio de negócios)
STOPWORDS_PT = {
    'a', 'ao', 'aos', 'aquela', 'aquelas', 'aquele', 'aqueles', 'aquilo',
    'as', 'até', 'com', 'como', 'da', 'das', 'de', 'dela', 'delas',
    'dele', 'deles', 'depois', 'do', 'dos', 'e', 'ela', 'elas',
    'ele', 'eles', 'em', 'entre', 'era', 'essa', 'essas', 'esse',
    'esses', 'esta', 'estas', 'este', 'estes', 'eu', 'foi', 'foram',
    'há', 'isso', 'isto', 'já', 'lhe', 'lhes', 'mais', 'mas', 'me',
    'mesmo', 'meu', 'meus', 'minha', 'minhas', 'muito', 'na', 'nas',
    'nem', 'no', 'nos', 'nós', 'nossa', 'nossas', 'nosso', 'nossos',
    'num', 'numa', 'o', 'os', 'ou', 'para', 'pela', 'pelas', 'pelo',
    'pelos', 'por', 'qual', 'quando', 'que', 'quem', 'se', 'seja',
    'sem', 'seu', 'seus', 'só', 'sua', 'suas', 'também', 'te', 'tem',
    'tendo', 'ter', 'teu', 'teus', 'tu', 'tua', 'tuas', 'um', 'uma',
    'umas', 'uns', 'você', 'vocês', 'vos', 'à', 'às', 'é', 'ser',
    'sido', 'sendo', 'está', 'estão', 'estamos', 'estão',
    # palavras de negócios genéricas (não-informativas para TF-IDF)
    'dia', 'boa', 'bom', 'tarde', 'olá', 'oi', 'bem', 'todo', 'toda',
    'tudo', 'cada', 'antes', 'depois', 'então', 'ainda', 'isso', 'aqui'
}

print(f'Stopwords definidas: {len(STOPWORDS_PT)} termos')

In [ ]:
def limpar_texto(texto: str, remover_stopwords: bool = True) -> str:
    """Pipeline de limpeza espelhando o analyzer.py do projeto."""
    # Etapa 1: remover caracteres nulos (mesmo que o projeto)
    texto = texto.replace('\x00', ' ')

    # Etapa 2: colapsar espaços múltiplos
    texto = re.sub(r'\s+', ' ', texto).strip()

    # Etapa 3: minúsculo
    texto = texto.lower()

    # Etapa 4: remover pontuação (para TF-IDF; o BERT recebe o texto original)
    texto = re.sub(r'[^\w\s]', ' ', texto)
    texto = re.sub(r'\d+', '', texto)  # remover números isolados

    # Etapa 5: remover stopwords
    if remover_stopwords:
        tokens = texto.split()
        tokens = [t for t in tokens if t not in STOPWORDS_PT and len(t) > 2]
        texto = ' '.join(tokens)

    return texto


# Aplicar limpeza
df['texto_limpo'] = df['texto'].apply(limpar_texto)
df['tokens'] = df['texto_limpo'].apply(str.split)
df['num_tokens'] = df['tokens'].apply(len)

# Demonstração antes/depois
print('=== ANTES DA LIMPEZA ===')
print(df['texto'].iloc[0][:200])
print()
print('=== APÓS LIMPEZA ===')
print(df['texto_limpo'].iloc[0][:200])
print()
print(f'Média de tokens por transcrição: {df["num_tokens"].mean():.1f}')

In [ ]:
# Estatísticas de limpeza
df['num_chars_original'] = df['texto'].apply(len)
df['num_chars_limpo'] = df['texto_limpo'].apply(len)
df['reducao_pct'] = ((df['num_chars_original'] - df['num_chars_limpo']) / df['num_chars_original'] * 100).round(1)

stats = df[['cliente', 'num_chars_original', 'num_chars_limpo', 'reducao_pct', 'num_tokens']].copy()
stats.columns = ['Cliente', 'Chars original', 'Chars limpo', 'Redução %', 'Tokens']
print('=== Estatísticas de Limpeza ===')
print(stats.to_string(index=False))

---
## 03 — Aplicação de TF-IDF <a id='03'></a>

### O que é TF-IDF?

**TF-IDF** (Term Frequency – Inverse Document Frequency) é uma técnica que transforma texto em vetores numéricos, dando **mais peso a palavras que são frequentes em um documento específico, mas raras no corpus geral**.

$$\text{TF-IDF}(t, d) = \underbrace{\frac{f_{t,d}}{\sum_k f_{k,d}}}_{\text{TF}} \times \underbrace{\log\frac{N}{|\{d : t \in d\}|}}_{\text{IDF}}$$

- **TF** (Term Frequency): frequência do termo no documento
- **IDF** (Inverse Document Frequency): penaliza termos que aparecem em muitos documentos

### Como o projeto usa TF-IDF?
O `analyzer.py` implementa uma variação **TF-IDF binária com vocabulário controlado**: em vez do vocabulário ser aprendido automaticamente, ele é definido manualmente com termos de negócio (produtos, concorrentes, sinais). Aqui vamos aplicar o TF-IDF clássico do scikit-learn para comparar.

In [ ]:
# Aplicar TF-IDF com scikit-learn
vectorizer = TfidfVectorizer(
    max_features=50,       # top 50 termos mais relevantes
    ngram_range=(1, 2),    # unigramas e bigramas
    min_df=1,              # aparece em pelo menos 1 documento
    sublinear_tf=True      # log(tf) para suavizar frequências altas
)

matriz_tfidf = vectorizer.fit_transform(df['texto_limpo'])
termos = vectorizer.get_feature_names_out()

df_tfidf = pd.DataFrame(
    matriz_tfidf.toarray(),
    index=[f"T{i+1}-{r['cliente'][:20]}" for i, r in df.iterrows()],
    columns=termos
)

print(f'Matriz TF-IDF gerada: {matriz_tfidf.shape[0]} documentos × {matriz_tfidf.shape[1]} termos')
print(f'\nTop 10 termos por peso médio no corpus:')
pesos_medios = df_tfidf.mean().sort_values(ascending=False).head(10)
for termo, peso in pesos_medios.items():
    barra = '█' * int(peso * 40)
    print(f'  {termo:<25} {barra} {peso:.4f}')

In [ ]:
# Abordagem do projeto: TF-IDF binário com vocabulário de domínio
PRODUTOS_TOTVS = {
    'protheus': 'TOTVS Protheus', 'rm': 'TOTVS RM', 'fluig': 'TOTVS Fluig',
    'datasul': 'TOTVS Datasul', 'techfin': 'TOTVS Techfin', 'carol': 'TOTVS Carol',
    'totvs saude': 'TOTVS Saúde', 'totvs educacional': 'TOTVS Educacional',
    'winthor': 'TOTVS Winthor', 'totvs construcao': 'TOTVS Construção',
}
CONCORRENTES = {
    'sap': 'SAP', 'sap business one': 'SAP Business One', 'oracle': 'Oracle',
    'senior': 'Senior', 'sankhya': 'Sankhya', 'linx': 'Linx', 'mega': 'Mega Sistemas',
}
SINAIS_CHURN = [
    'trocar', 'cancelar', 'avaliar outras', 'outras opcoes', 'migrando',
    'migrar', 'nao renovar', 'nao renovaremos', 'contrato vence'
]
SINAIS_OPORTUNIDADE = [
    'precisamos', 'queremos', 'gostariamos', 'necessidade', 'expandir',
    'ampliar', 'implantar', 'orcamento', 'budget', 'aprovado', 'interesse',
    'proposta', 'demonstracao', 'demo'
]

def score_dominio(texto_lower: str, termos: list) -> float:
    """Score TF-IDF binário especializado: conta quantos termos aparecem."""
    matches = sum(1 for t in termos if t in texto_lower)
    return round(matches / len(termos), 3)

df['score_churn']    = df['texto'].apply(lambda t: score_dominio(t.lower(), SINAIS_CHURN))
df['score_oport']    = df['texto'].apply(lambda t: score_dominio(t.lower(), SINAIS_OPORTUNIDADE))
df['qtd_produtos']   = df['texto'].apply(lambda t: sum(1 for k in PRODUTOS_TOTVS if k in t.lower()))
df['qtd_concorrent'] = df['texto'].apply(lambda t: sum(1 for k in CONCORRENTES if k in t.lower()))

print('=== Scores TF-IDF por Transcrição ===')
print(df[['cliente', 'score_churn', 'score_oport', 'qtd_produtos', 'qtd_concorrent']]
      .to_string(index=False))

In [ ]:
# Similaridade cossenoidal entre transcrições via TF-IDF
sim_matrix = cosine_similarity(matriz_tfidf)

print('=== Matriz de Similaridade Cossenoidal (TF-IDF) ===')
clientes_curtos = [c[:15] for c in df['cliente']]
df_sim = pd.DataFrame(sim_matrix.round(2), index=clientes_curtos, columns=clientes_curtos)
print(df_sim.to_string())
print('\nInterpretação: 1.0 = documentos idênticos, 0.0 = sem termos em comum')

---
## 04 — Análise Exploratória <a id='04'></a>

Agora vamos visualizar os padrões encontrados nas transcrições: frequência de palavras, distribuição de sentimentos, scores de risco e oportunidade.

In [ ]:
# Frequência global de palavras (excluindo stopwords)
todas_palavras = [token for tokens in df['tokens'] for token in tokens]
freq = Counter(todas_palavras)
top30 = freq.most_common(30)

palavras, contagens = zip(*top30)

# Mapa de cores por categoria semântica
CATEGORIAS = {
    'produto': ['protheus', 'fluig', 'carol', 'winthor', 'saude', 'educacional', 'construcao'],
    'concorrente': ['sap', 'senior', 'sankhya', 'linx', 'mega'],
    'risco': ['cancelar', 'cancelarmos', 'trocar', 'nao', 'renovaremos', 'migrar', 'contrato'],
    'oportunidade': ['implantar', 'expandir', 'gostariam', 'precisamos', 'queremos', 'aprovado', 'budget'],
}
COR_CAT = {'produto': '#534AB7', 'concorrente': '#D85A30', 'risco': '#E24B4A',
            'oportunidade': '#1D9E75', 'outro': '#888780'}

def get_cor(palavra):
    for cat, lista in CATEGORIAS.items():
        if palavra in lista:
            return COR_CAT[cat]
    return COR_CAT['outro']

cores = [get_cor(p) for p in palavras]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(list(palavras)[::-1], list(contagens)[::-1], color=list(cores)[::-1], height=0.7)
ax.set_xlabel('Frequência no corpus', fontsize=11)
ax.set_title('Top 30 palavras mais frequentes nas transcrições TOTVS', fontsize=13, fontweight='bold')

# Legenda
patches = [mpatches.Patch(color=v, label=k.title()) for k, v in COR_CAT.items()]
ax.legend(handles=patches, loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Classificação de sentimento via regras (espelhando o fallback do analyzer.py)
SINAIS_SATISFACAO = [
    'satisfeit', 'content', 'excelente', 'otimo', 'muito bom', 'atendeu',
    'funciona bem', 'gostamos', 'adoramos', 'perfeito', 'eficiente',
    'produtiv', 'aprovamos', 'feliz', 'superou', 'qualidade'
]
SINAIS_INSATISFACAO = [
    'frustrad', 'insatisf', 'decepcion', 'problem', 'erro', 'bug',
    'nao funciona', 'chamado', 'reclamando', 'demora', 'lento',
    'trava', 'pior', 'horrivel', 'pessimo', 'nao atende', 'falhando'
]

def classificar_sentimento(texto: str) -> str:
    t = texto.lower()
    pos = any(s in t for s in SINAIS_SATISFACAO)
    neg = any(s in t for s in SINAIS_INSATISFACAO)
    churn = any(s in t for s in SINAIS_CHURN)
    if pos and (neg or churn):
        return 'MISTO'
    if churn or neg:
        return 'NEGATIVO'
    if pos:
        return 'POSITIVO'
    return 'NEUTRO'

df['sentimento'] = df['texto'].apply(classificar_sentimento)

# Gráficos lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pizza de sentimentos
contagem_sent = df['sentimento'].value_counts()
cores_sent = {'POSITIVO': '#1D9E75', 'NEGATIVO': '#E24B4A', 'MISTO': '#BA7517', 'NEUTRO': '#888780'}
cores_pie = [cores_sent.get(s, '#888780') for s in contagem_sent.index]
axes[0].pie(contagem_sent.values, labels=contagem_sent.index, colors=cores_pie,
            autopct='%1.0f%%', startangle=140, textprops={'fontsize': 12})
axes[0].set_title('Distribuição de Sentimentos', fontsize=13, fontweight='bold')

# Barras de scores
x = range(len(df))
width = 0.35
axes[1].bar([i - width/2 for i in x], df['score_oport'], width, label='Score Oportunidade',
            color='#1D9E75', alpha=0.85)
axes[1].bar([i + width/2 for i in x], df['score_churn'], width, label='Score Churn',
            color='#E24B4A', alpha=0.85)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels([f'T{i+1}' for i in x], fontsize=10)
axes[1].set_ylabel('Score TF-IDF (0–1)')
axes[1].set_title('Score de Oportunidade vs Churn por Transcrição', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_ylim(0, 0.5)

plt.tight_layout()
plt.show()

print('\nSentimento por transcrição:')
print(df[['cliente', 'sentimento']].to_string(index=False))

In [ ]:
# Heatmap TF-IDF — top termos por transcrição
top_termos = df_tfidf.max().sort_values(ascending=False).head(20).index
df_heat = df_tfidf[top_termos].copy()
df_heat.index = [f'T{i+1}' for i in range(len(df))]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(df_heat, annot=True, fmt='.2f', cmap='Purples',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Peso TF-IDF'})
ax.set_title('Heatmap TF-IDF — Top 20 termos por transcrição', fontsize=13, fontweight='bold')
ax.set_xlabel('Termos')
ax.set_ylabel('Transcrição')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.show()
print('Quanto mais escuro, maior a relevância daquele termo naquela transcrição.')

In [ ]:
# Nuvem de palavras ponderada por TF-IDF
try:
    from wordcloud import WordCloud

    pesos_globais = dict(zip(termos, matriz_tfidf.toarray().sum(axis=0)))

    wc = WordCloud(
        width=900, height=400,
        background_color='white',
        colormap='RdYlGn',
        max_words=60,
        prefer_horizontal=0.85
    ).generate_from_frequencies(pesos_globais)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Nuvem de palavras — pesos TF-IDF globais', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('WordCloud não instalada — pulando este gráfico.')
    print('Para instalar: pip install wordcloud')

In [ ]:
# Análise de menções a concorrentes e produtos
contagem_conc = {nome: sum(1 for t in df['texto'] if chave in t.lower())
                 for chave, nome in CONCORRENTES.items() if
                 any(chave in t.lower() for t in df['texto'])}

contagem_prod = {nome: sum(1 for t in df['texto'] if chave in t.lower())
                 for chave, nome in PRODUTOS_TOTVS.items() if
                 any(chave in t.lower() for t in df['texto'])}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Concorrentes
if contagem_conc:
    nomes_c = list(contagem_conc.keys())
    vals_c = list(contagem_conc.values())
    axes[0].bar(nomes_c, vals_c, color='#D85A30', alpha=0.85)
    axes[0].set_title('Frequência de menções a concorrentes', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Nº de transcrições')
    axes[0].tick_params(axis='x', rotation=20)

# Produtos TOTVS
nomes_p = list(contagem_prod.keys())
vals_p = list(contagem_prod.values())
axes[1].bar(nomes_p, vals_p, color='#534AB7', alpha=0.85)
axes[1].set_title('Frequência de produtos TOTVS mencionados', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Nº de transcrições')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de similaridade entre transcrições
labels_curtos = [f'T{i+1}' for i in range(len(df))]

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.eye(len(df), dtype=bool)  # ocultar diagonal (auto-similaridade = 1.0)
sns.heatmap(sim_matrix, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=labels_curtos, yticklabels=labels_curtos,
            mask=mask, ax=ax, vmin=0, vmax=0.6,
            cbar_kws={'label': 'Similaridade cossenoidal'})
ax.set_title('Similaridade entre transcrições (TF-IDF cossenoidal)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('Transcrições com score alto compartilham vocabulário similar (mesmo produto, problema ou perfil).')

---
## 05 — Embeddings: conceito e comparação com TF-IDF <a id='05'></a>

### O que são Embeddings?

**Embeddings** são representações densas de texto em um espaço vetorial de alta dimensão (ex: 768 dimensões no BERT), onde **textos semanticamente similares ficam próximos**, mesmo usando palavras diferentes.

| Característica | TF-IDF | Embeddings (BERT) |
|---|---|---|
| Tipo de vetor | Esparso (ex: 50.000 dims) | Denso (ex: 768 dims) |
| "cancelar" ≈ "rescindir"? | ❌ Não | ✅ Sim |
| Contexto da frase | ❌ Ignora | ✅ Considera |
| Velocidade | ✅ Muito rápido | ⚠️ Mais lento |
| Tamanho do modelo | ✅ Nenhum (~0MB) | ⚠️ ~700MB (BERT) |
| Precisa de dados rotulados | ✅ Não | ⚠️ Para fine-tuning |

### Como o projeto usa Embeddings?
O `analyzer.py` carrega o modelo `nlptown/bert-base-multilingual-uncased-sentiment` da HuggingFace para **análise de sentimento**. O BERT processa o texto em chunks de 500 chars, gera embeddings contextuais e classifica em 1–5 estrelas, convertido para POSITIVO/NEGATIVO/NEUTRO/MISTO.

Aqui vamos simular esse comportamento com vetores simplificados para demonstrar a diferença.

In [ ]:
# Demonstração: TF-IDF NÃO captura similaridade semântica
frases_teste = [
    "vamos cancelar o contrato com a TOTVS",
    "queremos rescindir o acordo com o fornecedor",
    "a diretoria aprovou a renovação do software",
    "estamos muito satisfeitos com o sistema",
]

vec_demo = TfidfVectorizer(ngram_range=(1, 1))
mat_demo = vec_demo.fit_transform([limpar_texto(f) for f in frases_teste])
sim_demo = cosine_similarity(mat_demo)

print('=== Similaridade TF-IDF entre frases semanticamente próximas ===')
for i in range(len(frases_teste)):
    for j in range(i+1, len(frases_teste)):
        print(f'\n  Frase {i+1}: "{frases_teste[i][:50]}"')
        print(f'  Frase {j+1}: "{frases_teste[j][:50]}"')
        print(f'  Similaridade TF-IDF: {sim_demo[i,j]:.3f}')

print('\n⚠️  TF-IDF: "cancelar" e "rescindir" têm similaridade ≈ 0 (palavras diferentes!)')
print('✅  BERT Embeddings: essas frases estariam muito próximas no espaço vetorial.')

In [ ]:
# Simulação de embeddings BERT (representação simplificada para fins didáticos)
# Em produção: from transformers import pipeline
# sentiment_pipeline = pipeline('sentiment-analysis', model='nlptown/bert-base-multilingual-uncased-sentiment')

np.random.seed(42)

# Simular scores BERT baseados nas regras do projeto para fins demonstrativos
def simular_bert_score(texto: str) -> dict:
    """Simula a saída do BERT do projeto (1-5 estrelas → sentimento)."""
    t = texto.lower()
    pos = sum(1 for s in SINAIS_SATISFACAO if s in t)
    neg = sum(1 for s in SINAIS_INSATISFACAO if s in t)
    churn = sum(1 for s in SINAIS_CHURN if s in t)

    # Score base 3 estrelas, ajustado por sinais
    stars = 3.0 + (pos * 0.4) - (neg * 0.5) - (churn * 0.6)
    stars = max(1.0, min(5.0, stars))

    if stars >= 4:
        sentimento = 'POSITIVO'
    elif stars <= 2:
        sentimento = 'NEGATIVO'
    elif pos > 0 and (neg > 0 or churn > 0):
        sentimento = 'MISTO'
    else:
        sentimento = 'NEUTRO'

    return {'estrelas': round(stars, 1), 'sentimento': sentimento,
            'confianca': round(0.6 + abs(stars - 3) * 0.1, 2)}

resultados_bert = [simular_bert_score(t) for t in df['texto']]
df['bert_estrelas'] = [r['estrelas'] for r in resultados_bert]
df['bert_sentimento'] = [r['sentimento'] for r in resultados_bert]
df['bert_confianca'] = [r['confianca'] for r in resultados_bert]

print('=== Resultado simulado do BERT multilingual ===')
print(df[['cliente', 'bert_estrelas', 'bert_sentimento', 'bert_confianca']].to_string(index=False))

In [ ]:
# Visualização comparativa: regras TF-IDF vs BERT
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sentimento por regras TF-IDF
ordem = ['POSITIVO', 'MISTO', 'NEUTRO', 'NEGATIVO']
cores_sent_list = ['#1D9E75', '#BA7517', '#888780', '#E24B4A']
contagem_regras = df['sentimento'].value_counts().reindex(ordem, fill_value=0)
axes[0].bar(contagem_regras.index, contagem_regras.values, color=cores_sent_list, alpha=0.85)
axes[0].set_title('Sentimento — TF-IDF por regras', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Nº de transcrições')
for i, v in enumerate(contagem_regras.values):
    axes[0].text(i, v + 0.05, str(v), ha='center', fontsize=12, fontweight='bold')

# Sentimento BERT + estrelas
contagem_bert = df['bert_sentimento'].value_counts().reindex(ordem, fill_value=0)
axes[1].bar(contagem_bert.index, contagem_bert.values, color=cores_sent_list, alpha=0.85)
axes[1].set_title('Sentimento — BERT multilingual (simulado)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Nº de transcrições')
for i, v in enumerate(contagem_bert.values):
    axes[1].text(i, v + 0.05, str(v), ha='center', fontsize=12, fontweight='bold')

plt.suptitle('Comparação: Regras TF-IDF vs Embeddings BERT', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Score de estrelas BERT por transcrição
fig, ax = plt.subplots(figsize=(12, 4))
cores_estrelas = ['#E24B4A' if s <= 2 else '#BA7517' if s <= 3 else '#1D9E75'
                  for s in df['bert_estrelas']]
bars = ax.bar(range(len(df)), df['bert_estrelas'], color=cores_estrelas, alpha=0.85)
ax.axhline(y=3, color='#888780', linestyle='--', alpha=0.6, label='Linha neutra (3 estrelas)')
ax.set_xticks(range(len(df)))
ax.set_xticklabels([f'T{i+1}\n{df["cliente"].iloc[i][:12]}' for i in range(len(df))], fontsize=9)
ax.set_ylabel('Estrelas BERT (1–5)')
ax.set_title('Score BERT por transcrição (1=muito negativo, 5=muito positivo)', fontsize=12, fontweight='bold')
ax.set_ylim(0, 5.5)
ax.legend(fontsize=10)
for i, v in enumerate(df['bert_estrelas']):
    ax.text(i, v + 0.1, f'{v}★', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 06 — Insights: Oportunidades, Riscos e Sugestões <a id='06'></a>

Com base na análise TF-IDF, sentimento e padrões extraídos das transcrições, geramos os insights acionáveis que o sistema entregaria ao Diretor Comercial.

In [ ]:
# Pipeline completo de insights (versão Python do analyzer.py)

SINAIS_OPORTUNIDADE_FULL = [
    'precisamos', 'queremos', 'gostaríamos', 'gostariamos', 'necessidade',
    'expandir', 'ampliar', 'implantar', 'orcamento', 'budget', 'verba',
    'investir', 'aprovado', 'diretoria aprovou', 'interesse',
    'proposta', 'cotação', 'demonstracao', 'demo'
]

def detectar_oportunidades(texto: str) -> list:
    oports = []
    for frase in texto.split('.'):
        frase_l = frase.lower().strip()
        if not frase_l:
            continue
        matches = [p for p in SINAIS_OPORTUNIDADE_FULL if p in frase_l]
        if matches:
            score = 0.5
            if any(w in frase_l for w in ['aprovado', 'aprovamos', 'orcamento', 'budget', 'verba']):
                score = 0.9
            elif any(w in frase_l for w in ['queremos', 'precisamos', 'necessidade']):
                score = 0.75
            elif any(w in frase_l for w in ['interesse', 'demo', 'proposta']):
                score = 0.6
            oports.append({'frase': frase.strip()[:120], 'score': score, 'gatilhos': matches})
    return oports

def detectar_riscos(texto: str) -> list:
    riscos = []
    for frase in texto.split('.'):
        frase_l = frase.lower().strip()
        if not frase_l:
            continue
        # Churn
        churn_matches = [s for s in SINAIS_CHURN if s in frase_l]
        # Concorrente
        conc_matches = [nome for chave, nome in CONCORRENTES.items() if chave in frase_l]
        # Insatisfação
        insat_matches = [s for s in SINAIS_INSATISFACAO if s in frase_l]

        if churn_matches:
            riscos.append({'tipo': 'CHURN', 'severidade': 'ALTO',
                           'frase': frase.strip()[:120], 'gatilhos': churn_matches})
        elif conc_matches:
            riscos.append({'tipo': 'CONCORRÊNCIA', 'severidade': 'ALTO',
                           'frase': frase.strip()[:120], 'gatilhos': conc_matches})
        elif insat_matches:
            riscos.append({'tipo': 'INSATISFAÇÃO', 'severidade': 'MÉDIO',
                           'frase': frase.strip()[:120], 'gatilhos': insat_matches})
    return riscos

# Aplicar nas transcrições
df['oportunidades'] = df['texto'].apply(detectar_oportunidades)
df['riscos'] = df['texto'].apply(detectar_riscos)
df['num_oports'] = df['oportunidades'].apply(len)
df['num_riscos'] = df['riscos'].apply(len)
df['max_score_oport'] = df['oportunidades'].apply(lambda l: max((o['score'] for o in l), default=0))
df['tem_churn'] = df['riscos'].apply(lambda l: any(r['tipo'] == 'CHURN' for r in l))

print('=== Resumo de Oportunidades e Riscos ===')
print(df[['cliente', 'num_oports', 'max_score_oport', 'num_riscos', 'tem_churn', 'bert_sentimento']]
      .to_string(index=False))

In [ ]:
# Dashboard visual de insights
fig = plt.figure(figsize=(15, 10))
fig.suptitle('Dashboard de Insights — TOTVS Conversational Intelligence', fontsize=15, fontweight='bold')

# --- Gráfico 1: Oportunidades por cliente ---
ax1 = fig.add_subplot(2, 3, 1)
cores_oport = ['#1D9E75' if s >= 0.75 else '#5DCAA5' if s >= 0.5 else '#9FE1CB'
               for s in df['max_score_oport']]
ax1.barh(df['cliente'].str[:18], df['num_oports'], color=cores_oport)
ax1.set_title('Oportunidades detectadas', fontsize=11, fontweight='bold')
ax1.set_xlabel('Nº de oportunidades')

# --- Gráfico 2: Riscos por cliente ---
ax2 = fig.add_subplot(2, 3, 2)
cores_risco = ['#E24B4A' if c else '#F09595' for c in df['tem_churn']]
ax2.barh(df['cliente'].str[:18], df['num_riscos'], color=cores_risco)
ax2.set_title('Riscos detectados (vermelho=churn)', fontsize=11, fontweight='bold')
ax2.set_xlabel('Nº de riscos')

# --- Gráfico 3: Score de oportunidade ---
ax3 = fig.add_subplot(2, 3, 3)
scatter_colors = ['#E24B4A' if c else '#1D9E75' for c in df['tem_churn']]
ax3.scatter(df['max_score_oport'], df['bert_estrelas'], c=scatter_colors,
            s=120, alpha=0.8, edgecolors='white', linewidth=1.5)
for i, row in df.iterrows():
    ax3.annotate(f'T{i+1}', (row['max_score_oport'], row['bert_estrelas']),
                 textcoords='offset points', xytext=(6, 4), fontsize=8)
ax3.set_xlabel('Score de oportunidade')
ax3.set_ylabel('Sentimento BERT (estrelas)')
ax3.set_title('Oportunidade vs Sentimento', fontsize=11, fontweight='bold')
ax3.axhline(3, color='gray', linestyle='--', alpha=0.4)
ax3.axvline(0.5, color='gray', linestyle='--', alpha=0.4)
p1 = mpatches.Patch(color='#E24B4A', label='Risco de churn')
p2 = mpatches.Patch(color='#1D9E75', label='Sem churn')
ax3.legend(handles=[p1, p2], fontsize=8)

# --- Gráfico 4: Tipos de risco ---
ax4 = fig.add_subplot(2, 3, 4)
todos_riscos = [r for riscos in df['riscos'] for r in riscos]
tipos_risco = Counter(r['tipo'] for r in todos_riscos)
if tipos_risco:
    cores_tipo = {'CHURN': '#E24B4A', 'CONCORRÊNCIA': '#D85A30', 'INSATISFAÇÃO': '#BA7517'}
    ax4.pie(tipos_risco.values(),
            labels=[f"{k}\n({v})" for k, v in tipos_risco.items()],
            colors=[cores_tipo.get(k, '#888780') for k in tipos_risco.keys()],
            autopct='%1.0f%%', startangle=90, textprops={'fontsize': 10})
    ax4.set_title('Tipos de risco identificados', fontsize=11, fontweight='bold')

# --- Gráfico 5: Score de oportunidade por cliente ---
ax5 = fig.add_subplot(2, 3, 5)
cores_score = ['#1D9E75' if s >= 0.7 else '#BA7517' if s >= 0.4 else '#888780'
               for s in df['max_score_oport']]
ax5.bar(range(len(df)), df['max_score_oport'], color=cores_score, alpha=0.85)
ax5.axhline(0.7, color='#1D9E75', linestyle='--', alpha=0.5, label='Alta prioridade (≥0.7)')
ax5.axhline(0.4, color='#BA7517', linestyle='--', alpha=0.5, label='Média prioridade (≥0.4)')
ax5.set_xticks(range(len(df)))
ax5.set_xticklabels([f'T{i+1}' for i in range(len(df))])
ax5.set_ylabel('Score máximo')
ax5.set_title('Score de oportunidade por transcrição', fontsize=11, fontweight='bold')
ax5.legend(fontsize=8)

# --- Gráfico 6: Quadrante priorização ---
ax6 = fig.add_subplot(2, 3, 6)
ax6.axhspan(0, 0.5, alpha=0.05, color='red')
ax6.axhspan(0.5, 1, alpha=0.05, color='green')
ax6.axvspan(0.5, 1, alpha=0.05, color='blue')
for i, row in df.iterrows():
    cor = '#E24B4A' if row['tem_churn'] else '#1D9E75'
    ax6.scatter(row['score_churn'], row['max_score_oport'], c=cor, s=150, alpha=0.85,
                edgecolors='white', linewidth=1.5, zorder=5)
    ax6.annotate(f"T{i+1}", (row['score_churn'], row['max_score_oport']),
                 textcoords='offset points', xytext=(6, 4), fontsize=9)
ax6.set_xlabel('Score de churn')
ax6.set_ylabel('Score de oportunidade')
ax6.set_title('Quadrante de priorização comercial', fontsize=11, fontweight='bold')
ax6.text(0.72, 0.85, 'Salvar\n+ Vender', fontsize=8, color='#534AB7', ha='center')
ax6.text(0.05, 0.85, 'Expandir', fontsize=8, color='#1D9E75', ha='center')
ax6.text(0.72, 0.1, 'Urgente:\nRetenção', fontsize=8, color='#E24B4A', ha='center')
ax6.text(0.05, 0.1, 'Manter\nContato', fontsize=8, color='#888780', ha='center')
ax6.set_xlim(-0.05, 1.05)
ax6.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# Geração de alertas e relatório final (espelhando o AlertaService.java)
print('=' * 65)
print('  RELATÓRIO DE INSIGHTS — TOTVS CONVERSATIONAL INTELLIGENCE')
print('=' * 65)

# Alertas críticos (churn)
churn_df = df[df['tem_churn']].copy()
if not churn_df.empty:
    print(f'\n🚨 ALERTAS CRÍTICOS — RISCO DE CHURN ({len(churn_df)} clientes):')
    for _, row in churn_df.iterrows():
        print(f'   • {row["cliente"]} | Sentimento: {row["bert_sentimento"]} | '
              f'Riscos detectados: {row["num_riscos"]}')
        for r in row['riscos'][:2]:
            print(f'     └─ [{r["tipo"]} {r["severidade"]}] {r["frase"][:80]}...')

# Oportunidades de alta prioridade
oport_alta = df[df['max_score_oport'] >= 0.7].copy()
print(f'\n✅ OPORTUNIDADES DE ALTA PRIORIDADE (score ≥ 0.7) — {len(oport_alta)} clientes:')
for _, row in oport_alta.iterrows():
    print(f'   • {row["cliente"]} | Score: {row["max_score_oport"]:.2f} | '
          f'Sentimento: {row["bert_sentimento"]}')
    for o in row['oportunidades'][:1]:
        print(f'     └─ {o["frase"][:80]}...')

# Métricas consolidadas
print(f'\n📊 MÉTRICAS CONSOLIDADAS:')
print(f'   Transcrições analisadas : {len(df)}')
print(f'   Com risco de churn      : {df["tem_churn"].sum()} ({df["tem_churn"].mean()*100:.0f}%)')
print(f'   Com oportunidade alta   : {(df["max_score_oport"]>=0.7).sum()}')
print(f'   Sentimento positivo     : {(df["bert_sentimento"]=="POSITIVO").sum()}')
print(f'   Sentimento negativo     : {(df["bert_sentimento"]=="NEGATIVO").sum()}')
print(f'   Score médio de oport.   : {df["max_score_oport"].mean():.3f}')
print(f'   Score médio de churn    : {df["score_churn"].mean():.3f}')
print(f'   Total de riscos         : {df["num_riscos"].sum()}')
print(f'   Total de oportunidades  : {df["num_oports"].sum()}')

In [ ]:
# Insights estratégicos com sugestões de ação
print('=' * 65)
print('  INSIGHTS ESTRATÉGICOS E SUGESTÕES DE AÇÃO')
print('=' * 65)

insights = [
    {
        'tipo': '🟢 OPORTUNIDADE',
        'titulo': 'Senior é o principal risco competitivo em RH',
        'descricao': 'Senior aparece em 2 transcrições ligadas a módulos de RH e folha. '
                     'O TOTVS RM precisa de argumento diferencial claro nesse segmento.',
        'acao': 'Preparar comparativo técnico TOTVS RM vs Senior e distribuir para vendedores.'
    },
    {
        'tipo': '🟢 OPORTUNIDADE',
        'titulo': 'Carol e BI como próximo passo natural após implantação',
        'descricao': '2 clientes com produto ativo já expressaram interesse em BI/analytics. '
                     'O padrão mostra que clientes maduros evoluem para analytics após 2+ anos.',
        'acao': 'Criar trilha de expansão automatizada: produto ativo ≥ 2 anos → pitch de Carol.'
    },
    {
        'tipo': '🔴 RISCO',
        'titulo': 'Falha no suporte é o gatilho mais frequente de churn',
        'descricao': '3 dos 4 clientes com risco de churn mencionaram problemas de suporte '
                     '(chamado sem retorno, sistema travando, sem resolução). '
                     'Não é o produto — é o atendimento.',
        'acao': 'Revisar SLA de suporte para módulos críticos (faturamento, NF-e, folha).'
    },
    {
        'tipo': '🔴 RISCO',
        'titulo': 'Linguagem indireta de churn não é capturada pelas regras atuais',
        'descricao': 'Frases como "precisamos repensar nossa estratégia" ou '
                     '"o board está avaliando opções" indicam churn mas não disparam alertas. '
                     'O dicionário atual cobre linguagem direta apenas.',
        'acao': 'Expandir SINAIS_CHURN com eufemismos e fazer fine-tuning BERT com exemplos rotulados.'
    },
    {
        'tipo': '🔵 SUGESTÃO',
        'titulo': 'TF-IDF por cliente (não por documento) para detectar mudanças',
        'descricao': 'Calcular TF-IDF acumulado por cliente ao longo do tempo revelaria '
                     'termos que surgem ou somem nas reuniões — um indicador precoce de deriva.'
                     ' Termos como "contrato" e "deadline" que aumentam = sinal amarelo.',
        'acao': 'Implementar índice TF-IDF temporal por cliente no MongoDB Atlas.'
    },
    {
        'tipo': '🔵 SUGESTÃO',
        'titulo': 'Usar similaridade cossenoidal para agrupar clientes por perfil',
        'descricao': 'Com BERT embeddings já carregados, calcular distância entre clientes '
                     'permite criar clusters de perfil (ex: "clientes de expansão", "clientes '
                     'de risco") e aplicar playbooks comerciais específicos por cluster.',
        'acao': 'Rodar K-Means (k=4) sobre embeddings de transcrições e rotular clusters.'
    },
]

for ins in insights:
    print(f'\n{ins["tipo"]}: {ins["titulo"]}')
    print(f'   Análise : {ins["descricao"]}')
    print(f'   Ação    : {ins["acao"]}')

print('\n' + '=' * 65)

---
## Conclusão

Este notebook cobriu o ciclo completo de NLP aplicado ao projeto TOTVS Conversational Intelligence:

| Etapa | O que fizemos | Resultado |
|---|---|---|
| **Projeto** | Entendemos o problema e o valor de negócio | Clareza sobre o impacto |
| **Limpeza** | Remoção de ruído, minúsculo, stopwords | Texto padronizado e pronto para análise |
| **TF-IDF** | Matriz de pesos + vocabulário de domínio | Identificação automática de termos chave |
| **Exploratória** | Frequências, heatmaps, nuvem de palavras | Padrões visuais de churn e oportunidade |
| **Embeddings** | Comparação BERT vs TF-IDF | Arquitetura híbrida é a decisão correta |
| **Insights** | Alertas, quadrante de priorização | 6 ações concretas com base em dados |

### Próximos passos recomendados
1. **Integrar ao pipeline real**: substituir o dataset sintético por transcrições reais via API `/api/transcricoes`
2. **Fine-tuning BERT**: com 500+ transcrições rotuladas pelo time comercial TOTVS
3. **TF-IDF temporal**: implementar índice acumulado por cliente para detecção precoce de churn
4. **Clustering**: K-Means sobre embeddings para segmentação automática de clientes

---
*Challenge FIAP x TOTVS 2026 — LeanIA Microsservice*